# Test Pipeline Mooré — Meta MMS

**But:** Tester la qualité de la reconnaissance et synthèse vocale en Mooré

Pipeline testé:
- **TTS** (Texte → Voix): `facebook/mms-tts-mos`
- **ASR** (Voix → Texte): `facebook/mms-1b-all` avec adapter Mooré
- **Traduction** (Mooré ↔ Français): `facebook/nllb-200-distilled-600M`

> Active le GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
# Installe les dépendances
!pip install transformers torch scipy soundfile accelerate -q
print('Installation terminée')

## ÉTAPE 1 — TTS : Générer voix Mooré depuis texte

In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch
import scipy.io.wavfile as wavfile
import numpy as np
from IPython.display import Audio, display

print('Chargement modèle TTS Mooré...')
tokenizer = AutoTokenizer.from_pretrained('facebook/mms-tts-mos')
tts_model = VitsModel.from_pretrained('facebook/mms-tts-mos')
print('Modèle TTS chargé!')

In [ ]:
# Phrases Mooré à tester
# (modifiables - ajoute tes propres phrases)
phrases = [
    ('Laafi bala?',       'Comment vas-tu?'),
    ('Laafi',             'Bien / Paix'),
    ('Baraka',            'Merci'),
    ('Yaa sooma',         'Bonjour (matin)'),
    ('M soaba',           'Mon ami'),
]

sampling_rate = tts_model.config.sampling_rate
generated_audios = {}

for moore_text, french_meaning in phrases:
    inputs = tokenizer(moore_text, return_tensors='pt')
    with torch.no_grad():
        output = tts_model(**inputs).waveform
    audio = output.squeeze().numpy()
    generated_audios[moore_text] = audio
    duration = len(audio) / sampling_rate
    print(f"Mooré: '{moore_text}' | Français: '{french_meaning}' | Durée: {duration:.2f}s")
    display(Audio(audio, rate=sampling_rate))

print('\nÉCOUTE LES FICHIERS AUDIO CI-DESSUS')
print('La voix sonne-t-elle naturelle en Mooré?')

## ÉTAPE 2 — ASR : Transcrire voix Mooré en texte

In [ ]:
from transformers import Wav2Vec2ForCTC, AutoProcessor

print('Chargement modèle ASR MMS-1B...')
print('(~4GB - prend 2-3 minutes)')

processor = AutoProcessor.from_pretrained('facebook/mms-1b-all')
asr_model = Wav2Vec2ForCTC.from_pretrained('facebook/mms-1b-all')

# Activer langue Mooré
processor.tokenizer.set_target_lang('mos')
asr_model.load_adapter('mos')

print('Modèle ASR chargé avec adapter Mooré!')

In [ ]:
print('=== RÉSULTATS ASR ===')
print(f'{"Phrase originale":<20} | {"Transcription ASR":<25} | Qualité')
print('-' * 65)

for moore_text, audio in generated_audios.items():
    inputs = processor(audio, sampling_rate=sampling_rate, return_tensors='pt')
    with torch.no_grad():
        logits = asr_model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.decode(predicted_ids[0])

    # Score basique
    original_words = set(moore_text.lower().replace('?', '').split())
    transcribed_words = set(transcription.lower().split())
    overlap = len(original_words & transcribed_words) / max(len(original_words), 1)
    quality = 'BON' if overlap > 0.5 else 'MOYEN' if overlap > 0.2 else 'FAIBLE'

    print(f"{moore_text:<20} | {transcription:<25} | {quality} ({overlap*100:.0f}%)")

print('\nNote: ASR sur audio TTS = cas idéal.')
print('Qualité réelle sur voix humaine sera différente (généralement meilleure pour voix native).')

## ÉTAPE 3 — Traduction Mooré ↔ Français (NLLB)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer as NLLBTokenizer

print('Chargement modèle NLLB-200 (traduction)...')
nllb_model_id = 'facebook/nllb-200-distilled-600M'
nllb_tokenizer = NLLBTokenizer.from_pretrained(nllb_model_id)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_model_id)
print('NLLB chargé!')

def translate(text, src_lang, tgt_lang):
    nllb_tokenizer.src_lang = src_lang
    inputs = nllb_tokenizer(text, return_tensors='pt')
    target_lang_id = nllb_tokenizer.lang_code_to_id[tgt_lang]
    with torch.no_grad():
        output = nllb_model.generate(**inputs, forced_bos_token_id=target_lang_id, max_length=100)
    return nllb_tokenizer.decode(output[0], skip_special_tokens=True)

# Test Mooré -> Français
print('\n=== TRADUCTION Mooré → Français ===')
moore_sentences = [
    'Laafi bala?',
    'Baraka',
    'M soaba',
]
for s in moore_sentences:
    result = translate(s, 'mos_Latn', 'fra_Latn')
    print(f"  Mooré: '{s}' → Français: '{result}'")

# Test Français -> Mooré
print('\n=== TRADUCTION Français → Mooré ===')
french_sentences = [
    'Bonjour, comment allez-vous?',
    'Merci beaucoup',
    'Je veux apprendre la robotique',
]
for s in french_sentences:
    result = translate(s, 'fra_Latn', 'mos_Latn')
    print(f"  Français: '{s}' → Mooré: '{result}'")

## ÉTAPE 4 — Pipeline Complet (voix Mooré → Claude → voix Mooré)

In [ ]:
# Simule le pipeline complet sans Claude (remplacé par réponse statique)
# Pour prod: remplacer MOCK_CLAUDE_RESPONSE par appel API Claude

def pipeline_moore_to_moore(moore_audio, sampling_rate):
    print('1. ASR: Voix Mooré → Texte Mooré')
    inputs = processor(moore_audio, sampling_rate=sampling_rate, return_tensors='pt')
    with torch.no_grad():
        logits = asr_model(**inputs).logits
    moore_text = processor.decode(torch.argmax(logits, dim=-1)[0])
    print(f'   Texte détecté: "{moore_text}"')

    print('2. Traduction: Mooré → Français')
    french_text = translate(moore_text, 'mos_Latn', 'fra_Latn')
    print(f'   Français: "{french_text}"')

    print('3. Claude API (simulé)')
    # TODO: Remplacer par vrai appel Claude
    claude_response_fr = f'[Réponse Claude en français à: {french_text}]'
    print(f'   Réponse: "{claude_response_fr}"')

    print('4. Traduction: Français → Mooré')
    moore_response = translate(claude_response_fr, 'fra_Latn', 'mos_Latn')
    print(f'   Mooré: "{moore_response}"')

    print('5. TTS: Texte Mooré → Voix Mooré')
    inputs_tts = tokenizer(moore_response, return_tensors='pt')
    with torch.no_grad():
        audio_out = tts_model(**inputs_tts).waveform.squeeze().numpy()
    print('   Audio généré!')

    return audio_out, moore_text, french_text, moore_response

# Test avec 'Laafi bala?'
print('=== TEST PIPELINE COMPLET ===')
print('Input: audio de "Laafi bala?" (généré par TTS)\n')
test_audio = generated_audios['Laafi bala?']
audio_out, detected, french, response_moore = pipeline_moore_to_moore(test_audio, sampling_rate)

print('\nRésultat final (audio réponse en Mooré):')
display(Audio(audio_out, rate=sampling_rate))

## Résumé Qualité

Après avoir tourné tous les tests, évaluer:

| Composant | Qualité observée | Note /5 |
|-----------|-----------------|--------|
| TTS (voix générée) | ? | ? |
| ASR (transcription) | ? | ? |
| Traduction Mooré→FR | ? | ? |
| Traduction FR→Mooré | ? | ? |

**Décision:** Si score moyen ≥ 3/5 → pipeline viable pour MVP

**Next steps après test:**
1. Collecter audio Mooré natif pour fine-tuning
2. Scraper YouTube vidéos en Mooré
3. Construire app Android avec ce pipeline